# Text Preprocessing Pipeline

Applies the full preprocessing pipeline to all six Amazon review files:
- Drops constant and low-value columns
- Handles missing values (critical vs non-critical fields)
- Cleans HTML tags and entities from review text
- Combines `review_headline` + `review_body` into a single `review` column
- Derives `sentiment_label`, `helpfulness_ratio`, and date-based features
- Saves each processed file as Parquet to `data/processed/`

In [ ]:
import os
import re
import html
import pandas as pd

DATA_DIR  = "../data"
OUT_DIR   = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {
    "Apparel":   "Apparel_review.tsv",
    "Beauty":    "Beauty_review.tsv",
    "Books":     "Books_review.tsv",
    "Furniture": "Furniture_review.tsv",
    "Mobile":    "Mobile_review.tsv",
    "Outdoors":  "Outdoors_review.tsv",
}

# Fields where a missing value means the row must be dropped
CRITICAL_FIELDS = ["product_id", "product_parent", "product_category",
                   "star_rating", "review_body", "review_date"]

## Text Cleaning Function

In [ ]:
def clean_text(text):
    """Remove HTML tags/entities, lowercase, normalise whitespace."""
    if not isinstance(text, str) or not text.strip():
        return None
    text = html.unescape(text)                  # decode &#34; &#39; etc.
    text = re.sub(r"<[^>]+>", " ", text)        # strip HTML tags
    text = text.lower()
    text = re.sub(r"[\t\n\r]+", " ", text)      # collapse line breaks / tabs
    text = re.sub(r" {2,}", " ", text).strip()  # collapse extra spaces
    return text if text else None

## Preprocessing Pipeline

In [ ]:
def preprocess(df, source_category):
    # --- source category (from filename, more reliable than product_category column) ---
    df["source_category"] = source_category

    # --- drop constant column ---
    df = df.drop(columns=["marketplace"])

    # --- drop rows missing critical fields ---
    df = df.dropna(subset=CRITICAL_FIELDS)
    df = df[df["review_body"].str.strip().ne("")]  # drop empty review_body

    # --- missing values: non-critical text → None, numeric → NaN ---
    df["review_headline"] = df["review_headline"].where(df["review_headline"].notna(), other=None)
    df["customer_id"]     = df["customer_id"].where(df["customer_id"].notna(), other=float("nan"))

    # --- text cleaning ---
    df["review_headline"] = df["review_headline"].apply(clean_text)
    df["review_body"]     = df["review_body"].apply(clean_text)

    # drop rows where review_body became None after cleaning
    df = df.dropna(subset=["review_body"])

    # --- combine headline + body into review ---
    df["review"] = df.apply(
        lambda r: (r["review_headline"] + ". " + r["review_body"])
                  if r["review_headline"] else r["review_body"],
        axis=1,
    )
    df = df.drop(columns=["review_headline", "review_body"])

    # --- sentiment label ---
    def sentiment(rating):
        if rating <= 2: return "negative"
        if rating == 3: return "neutral"
        return "positive"

    df["sentiment_label"] = df["star_rating"].apply(sentiment)
    df = df.drop(columns=["star_rating"])

    # --- helpfulness ratio ---
    df["helpfulness_ratio"] = df.apply(
        lambda r: r["helpful_votes"] / r["total_votes"] if r["total_votes"] > 0 else 0,
        axis=1,
    )

    # --- date features ---
    dates = pd.to_datetime(df["review_date"], errors="coerce")
    df["review_year"]        = dates.dt.year
    df["review_month"]       = dates.dt.month
    df["review_day_of_week"] = dates.dt.day_name()

    return df

## Run Pipeline & Save

In [ ]:
for category, filename in FILES.items():
    print(f"Processing {category}...")

    df = pd.read_csv(
        os.path.join(DATA_DIR, filename),
        sep="\t",
        on_bad_lines="skip",
        engine="python",
    )
    rows_before = len(df)

    df = preprocess(df, category)

    out_path = os.path.join(OUT_DIR, f"{category.lower()}_processed.parquet")
    df.to_parquet(out_path, index=False)

    print(f"  {rows_before:>9,} → {len(df):>9,} rows  ({rows_before - len(df):,} dropped)")
    print(f"  Saved to {out_path}")
    print()

## Verify Output

Quick sanity check on one processed file.

In [ ]:
sample = pd.read_parquet(os.path.join(OUT_DIR, "apparel_processed.parquet"))
print(sample.shape)
print(sample.dtypes)
print()
sample[["review_id", "sentiment_label", "helpfulness_ratio",
        "review_year", "review_month", "review_day_of_week", "review"]].head(3)

In [ ]:
# Sentiment label distribution
sample["sentiment_label"].value_counts()